## NLP Final
### Part 3: Named Entity Recognition - Analysis
### Aren Mizuno
### March 12, 2026

In [2]:
# Imports
import pandas as pd
from google.colab import drive

In [3]:
# Mount drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Load data ner
ner_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/ner.parquet"
ner = pd.read_parquet(ner_path, engine="pyarrow")
ner.head(2)

,url,date,title,text,text_for_extraction,text_clean,company,technology
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","bad idea ai price (bad), market cap, price tod...",[Bad Idea AI Price],[]
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,this ai video of gymnastics might be the freak...,"[Instagram, Trump, Werners AI Art]",[]


In [5]:
# Load data bertopic text
bertopic_text_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_text.parquet"
bertopic_text = pd.read_parquet(bertopic_text_path, engine="pyarrow")
bertopic_text.head(2)

,url,date,title,text,bertopic_topic_text
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",bad idea ai price bad market cap price today c...,10
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,this ai video of gymnastics might be the freak...,0


In [6]:
# Companies
companies_exploded = ner["company"].explode().dropna()

company_counts = companies_exploded.value_counts()

print("Top Companies:")
print(company_counts.head(20))

print("\nTotal unique companies:", company_counts.shape[0])
print("Total company mentions:", companies_exploded.shape[0])

Top Companies:
company
Google                     21352
Microsoft                  15467
Amazon                     10361
Apple                       8024
Instagram                   6570
Meta                        6280
Nvidia                      5139
Facebook                    4954
Nexstar Media Inc           2760
Tesla                       2530
Samsung                     2511
Intel                       2334
Artificial Intelligence     2249
Generative AI               2224
Cision US Inc               2196
Newsmatics Inc              2175
Gray Media Group            2151
Gray Television, Inc        2118
Research                    2101
IBM                         2088
Name: count, dtype: int64

Total unique companies: 71885
Total company mentions: 357926


In [7]:
# Technologies
tech_exploded = ner["technology"].explode().dropna()

tech_counts = tech_exploded.value_counts()

print("\nTop Technologies:")
print(tech_counts.head(20))

print("\nTotal unique technologies:", tech_counts.shape[0])
print("Total technology mentions:", tech_exploded.shape[0])


Top Technologies:
technology
artificial intelligence           84072
generative ai                     33503
chatbot                           27612
ai model                          24618
large language model              20954
machine learning                  19631
robotics                           6864
deep learning                      4369
natural language processing        4268
computer vision                    3314
neural network                     3149
foundation model                   2212
fine-tuning                        1522
transformer                        1436
reinforcement learning             1118
retrieval-augmented generation     1062
autonomous driving                  965
prompt engineering                  923
multimodal ai                       515
embeddings                          284
Name: count, dtype: int64

Total unique technologies: 25
Total technology mentions: 243387


In [8]:
# Topic → label mapping
topic_map = {
    -1: "Cross-Industry AI News / Outliers",
    0: "Enterprise Technology",
    1: "Consumer & Public Services",
    2: "Financial Services & Business",
    3: "Corporate AI Announcements",
    4: "Enterprise Software & Cybersecurity",
    5: "Healthcare & Life Sciences",
    6: "Financial Markets & Investing",
    7: "Media, Journalism & Public Policy",
    8: "Business Operations & Enterprise Automation",
    9: "Workforce & Compensation",
    10: "Cryptocurrency & Digital Assets",
    11: "Publishing & Intellectual Property",
    12: "Public AI Companies",
    13: "Technology Industry News",
    14: "Music & Entertainment",
    15: "African Technology Development",
    16: "AI Market Commentary"
}

# Replace topic numbers with labels
bertopic_text["topic_label"] = bertopic_text["bertopic_topic_text"].map(topic_map)

In [9]:
# Merge dfs
merged = bertopic_text[["url", "topic_label"]].merge(
    ner[["url", "company", "technology"]],
    on="url",
    how="inner"
)

print("Merged shape:", merged.shape)
display(merged.head(2))

Merged shape: (136233, 4)


,url,topic_label,company,technology
0,https://blockworks.co/price/bad,Cryptocurrency & Digital Assets,[Bad Idea AI Price],[]
1,https://boingboing.net/2024/07/01/this-ai-vide...,Enterprise Technology,"[Instagram, Trump, Werners AI Art]",[]


In [10]:
# Top 10 companies and technology by topic
for topic in sorted(merged["topic_label"].dropna().unique()):
    print("\n" + "="*60)
    print(f"TOPIC {topic}")
    print("="*60)

    top_comp = (
        merged.loc[merged["topic_label"] == topic, "company"]
        .explode()
        .dropna()
        .value_counts()
        .head(10)
    )

    top_tech = (
        merged.loc[merged["topic_label"] == topic, "technology"]
        .explode()
        .dropna()
        .value_counts()
        .head(10)
    )

    print("\nTop Companies:")
    print(top_comp)

    print("\nTop Technologies:")
    print(top_tech)


TOPIC AI Market Commentary

Top Companies:
company
My European Quotes    446
Motley                269
Nvidia                261
Amazon                170
Microsoft             134
Alphabet              122
TradeTalks            116
AMD                    63
Nasdaq, Inc            61
Google                 60
Name: count, dtype: int64

Top Technologies:
technology
artificial intelligence    664
generative ai              187
ai model                   168
large language model       123
machine learning            88
chatbot                     63
robotics                    58
autonomous driving          21
foundation model            15
computer vision             12
Name: count, dtype: int64

TOPIC African Technology Development

Top Companies:
company
Google                     176
Arena Holdings             114
Microsoft                   98
Facebook                    62
Apple                       52
Twitter                     43
Meta                        41
Instagram        